In [1]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q sentencepiece protobuf
!pip install -q pillow matplotlib seaborn pandas numpy tqdm scikit-learn
!pip install -U transformers
!pip install -U datasets
!pip install -U peft
!pip install -U trl
!pip install -U accelerate
!pip install -U bitsandbytes
!pip install -U pillow
!pip install -U sentencepiece
!pip install -U qwen-vl-utils

In [2]:
import os
from datasets import load_dataset

# ============================================================
# Output Folder
# ============================================================
OUTPUT_DIR = "preprocessed_dataset/"

# ============================================================
# Load Dataset from Hugging Face
# ============================================================

ds = load_dataset("MLforHealthcare/mimic-cxr")

train_data = ds["train"]
validation_data = ds["validation"]
test_data = ds["test"]

print("Dataset Loaded Successfully!")

# ============================================================
# Check Dataset Information
# ============================================================

print(f"Training Samples   : {len(train_data)}")
print(f"Validation Samples : {len(validation_data)}")
print(f"Test Samples       : {len(test_data)}")

Dataset Loaded Successfully!
Training Samples   : 21443
Validation Samples : 4594
Test Samples       : 4596


In [3]:
import os
import shutil
from PIL import Image
from tqdm.auto import tqdm

# Make sure to define your base output directory
OUTPUT_DIR = "processed_dataset/"


def preprocess_and_save(dataset, split_name):
    split_dir = os.path.join(OUTPUT_DIR, split_name)

    # Delete old folder if it exists
    if os.path.exists(split_dir):
        shutil.rmtree(split_dir)

    # Create folders
    image_dir = os.path.join(split_dir, "images")
    report_dir = os.path.join(split_dir, "reports")

    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    print(f"\nProcessing {split_name} ({len(dataset)} samples)...")

    # Single, clean loop with progress bar
    for idx, sample in enumerate(tqdm(dataset, desc=split_name)):
        try:
            # ====================================================
            # Image Processing
            # ====================================================
            image = sample["image"]

            # If image is a file path, open it
            if not isinstance(image, Image.Image):
                image = Image.open(image)

            # Convert to RGB
            image = image.convert("RGB")

            # Resize
            image = image.resize((224, 224))

            # Save image
            image.save(
                os.path.join(image_dir, f"{idx:06d}.png"),
                format="PNG"
            )

            # ====================================================
            # Report Processing
            # ====================================================
            # MIMIC-CXR / HF datasets usually use 'report' or 'text'
            report = sample.get("reports") or sample.get("report") or ""

            with open(
                os.path.join(report_dir, f"{idx:06d}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(str(report))

        except Exception as e:
            print(f"\nError at sample {idx}: {e}")

    print(f"{split_name} completed.")
# ============================================================
# Execute Preprocessing for All Splits
# ============================================================

# Process Training Set
preprocess_and_save(train_data, "train")

# Process Validation Set
preprocess_and_save(validation_data, "validation")

# Process Test Set
preprocess_and_save(test_data, "test")



Processing train (21443 samples)...


train:   0%|          | 0/21443 [00:00<?, ?it/s]

train completed.

Processing validation (4594 samples)...


validation:   0%|          | 0/4594 [00:00<?, ?it/s]

validation completed.

Processing test (4596 samples)...


test:   0%|          | 0/4596 [00:00<?, ?it/s]

test completed.


In [4]:
print(ds)
print("Train:", len(ds["train"]))
print("Validation:", len(ds["validation"]))
print("Test:", len(ds["test"]))

DatasetDict({
    train: Dataset({
        features: ['image', 'reports'],
        num_rows: 21443
    })
    validation: Dataset({
        features: ['image', 'reports'],
        num_rows: 4594
    })
    test: Dataset({
        features: ['image', 'reports'],
        num_rows: 4596
    })
})
Train: 21443
Validation: 4594
Test: 4596


In [6]:
from pathlib import Path
print("Training Images :", len(list(Path("processed_dataset/train/images").glob("*.png"))))
print("Training Reports:", len(list(Path("processed_dataset/train/reports").glob("*.txt"))))

print("Validation Images :", len(list(Path("processed_dataset/validation/images").glob("*.png"))))
print("Validation Reports:", len(list(Path("processed_dataset/validation/reports").glob("*.txt"))))

print("Test Images :", len(list(Path("processed_dataset/test/images").glob("*.png"))))
print("Test Reports:", len(list(Path("processed_dataset/test/reports").glob("*.txt"))))

Training Images : 21443
Training Reports: 21443
Validation Images : 4594
Validation Reports: 4594
Test Images : 4596
Test Reports: 4596
